# **Classificação: Random Forest**
Prever tipo de propriedade a partir das características

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

clean_df = pd.read_csv("dataset_tratado.csv")

features = ['price', 'area', 'bedrooms', 'baths', 'city']
target = 'property_type'

X = clean_df[features]
y = clean_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Conjunto de treino:", X_train.shape)
print("Conjunto de teste:", X_test.shape, "\n")

### One hot encoding e preparação do modelo

In [ ]:
categorical = ['city']
numeric = ['price', 'area', 'bedrooms', 'baths']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('num', 'passthrough', numeric)
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=20,
        random_state=42,
        class_weight='balanced'
    ))
])

### Treinamento e scores por tipo de propriedade 

In [ ]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_, normalize='true')

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=model.classes_,
    yticklabels=model.classes_
)
plt.title('Matriz de Confusão Normalizada - Classificação de Tipo de Imóvel')
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

importances = model.named_steps['classifier'].feature_importances_
feature_names = model.named_steps['preprocessor'].get_feature_names_out()

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10,5))
sns.barplot(data=feat_imp.head(10), x='importance', y='feature')
plt.title('Top 10 Features Mais Importantes')
plt.tight_layout()
plt.show()

# Exportar previsões com dados reais
results = X_test.copy()
results['real_type'] = y_test.values
results['predicted_type'] = y_pred

results.head(50)
